In [0]:
from pyspark.sql.functions import *
from dateutil import parser

# ---------------- CONFIG ----------------
CATALOG = "chatbot_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Ensure silver schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

df = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.medical_history")

clean_date_format = (
    df.withColumn(
    "ongoing_treatment",
    when(col("ongoing_treatment").isin("Yes", "Y"), "Y").when(col("ongoing_treatment").isin("No", "N"), "N"))
    .withColumn("diagnosis_date",to_date(col("diagnosis_date"))).drop("_rescued_data")
)

# Removes duplicates
clean_dedup = clean_date_format.dropDuplicates(["patient_id"])

(clean_dedup.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(f"{CATALOG}.silver.medical_history")
)